# 📊 Binary Logistic Regression — Predicting High Wine Quality

Building, training, and evaluating a **Binary Logistic Regression** model to predict
whether a red wine is **high quality** (`is_high_quality = 1`, quality ≥ 7).

| | |
|---|---|
| **Target variable (Y)** | `is_high_quality` (1 = quality ≥ 7, 0 = quality < 7) |
| **Predictor variables (X)** | `alcohol`, `volatile acidity`, `sulphates`, `citric acid`, `fixed acidity`, `density`, `pH`, `chlorides`, `residual sugar`, `free sulfur dioxide`, `total sulfur dioxide` |
| **Model** | `LogisticRegression` (scikit-learn) |
| **Validation strategy** | Three splits: 70/30 · 40/60 · 80/20 (train + test evaluation) |
| **Class imbalance handling** | `class_weight='balanced'` |

> **Dataset:** UCI Wine Quality — 1,599 red wine samples with physicochemical properties and a sensory score.

> **Note on resampling:** no oversampling (`resample`) is applied in this notebook. Logistic Regression has
> built-in L2 regularization, a linear decision boundary, and only 11 coefficients, so on ~1,600 samples its
> overfitting risk is very low — the train/test evaluation below confirms this. Class imbalance (a separate
> issue from overfitting) is handled with `class_weight='balanced'`. Oversampling with `resample` is reserved
> for the CART tree notebooks, where overfitting is a genuine concern.

## I. Initial Setup — Imports and Visual Style

In [ ]:
# ── CELL 1: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

# Consistent visual style across the project
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print("✅ Libraries imported successfully")
print(f"   pandas   {pd.__version__}")
print(f"   numpy    {np.__version__}")
print(f"   seaborn  {sns.__version__}")

## II. Loading Data from UCI

We load the `winequality-red.csv` dataset and directly build the binary target
`is_high_quality` from the `quality` column.

**Binarization criterion:**
- `quality >= 7` → `is_high_quality = 1` (high quality wine)
- `quality < 7`  → `is_high_quality = 0` (standard or low quality wine)

In [ ]:
# ── CELL 2: Load data from UCI ────────────────────────────────────────────────
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

wine_quality = pd.read_csv(URL, sep=";")

# Build binary target: quality >= 7 → high quality
wine_quality['is_high_quality'] = (wine_quality['quality'] >= 7).astype(int)

print(f"✅ Data loaded: {wine_quality.shape[0]:,} rows × {wine_quality.shape[1]} columns")
print(f"\nAvailable columns:\n{wine_quality.columns.tolist()}")
print(f"\nTarget variable balance:")
bal = wine_quality['is_high_quality'].value_counts()
pct = wine_quality['is_high_quality'].value_counts(normalize=True) * 100
print(f"   High quality (quality >= 7) -> class 1: {bal[1]:,}  ({pct[1]:.1f}%)")
print(f"   Other quality (quality < 7) -> class 0: {bal[0]:,}  ({pct[0]:.1f}%)")
print(f"\nOriginal quality distribution:")
print(wine_quality['quality'].value_counts().sort_index())
wine_quality.head()

In [ ]:
# ── CELL 3: Validation and Exploration ────────────────────────────────────────
print("=" * 55)
print("DATASET VALIDATION")
print("=" * 55)

print("\n📋 Data types and missing values:")
print(wine_quality.dtypes.to_frame('dtype').join(
    wine_quality.isnull().sum().to_frame('nulls')
))

balance = wine_quality['is_high_quality'].value_counts()
pct     = wine_quality['is_high_quality'].value_counts(normalize=True) * 100
print(f"\n🎯 Target variable balance:")
print(f"   High quality  (1): {balance[1]:>5,}  ({pct[1]:.1f}%)")
print(f"   Other quality (0): {balance[0]:>5,}  ({pct[0]:.1f}%)")

print(f"\n📊 Means per class (high quality vs rest):")
features_num = ['alcohol', 'volatile acidity', 'sulphates',
                'citric acid', 'fixed acidity', 'pH', 'density']
print(wine_quality.groupby('is_high_quality')[features_num].mean().round(3).to_string())

## III. Data Preprocessing

### Applied steps
1. **Null check** — the UCI dataset contains no missing values.
2. **Scaling** — `StandardScaler` over all continuous numeric variables.
3. **X / y definition** — all physicochemical variables as predictors.

> Scaling is required for Logistic Regression so coefficients are comparable across variables.

In [ ]:
# ── CELL 4: Preprocessing ─────────────────────────────────────────────────────
df = wine_quality.copy()

scale_cols = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol'
]

drop_cols = ['quality', 'is_high_quality']
X_raw = df.drop(columns=drop_cols)
y     = df['is_high_quality']

scaler   = StandardScaler()
X_scaled = X_raw.copy()
X_scaled[scale_cols] = scaler.fit_transform(X_raw[scale_cols])

print("✅ Preprocessing completed")
print(f"   Shape X_raw    : {X_raw.shape}")
print(f"   Shape X_scaled : {X_scaled.shape}")
print(f"   Final columns: {X_scaled.columns.tolist()}")
print(f"\n   Classes in y -> 0: {(y==0).sum():,}  |  1: {(y==1).sum():,}")
print("\n⚠️  StandardScaler applied — Logistic Regression requires scaling")
print("   so that coefficients are comparable across variables.")

## IV. Training and Evaluation — Three Splits (Train + Test)

No oversampling is used here (see the note in the header). For each split (70/30, 40/60, 80/20):
1. Split `X_scaled` and `y` into train/test with `stratify=y`.
2. Train `LogisticRegression` with `class_weight='balanced'` to handle the class imbalance.
3. Evaluate on **both** train and test.
4. Compute the 5 required metrics: Accuracy, Precision, Recall, F1-Score, ROC-AUC.

Reporting train **and** test metrics lets us measure the train–test gap, the standard signal of overfitting.
For Logistic Regression this gap is expected to be small.

In [ ]:
# ── CELL 5: Train + evaluate function ─────────────────────────────────────────
def train_evaluate(X, y, test_size, random_state=42, name="Split"):
    """
    Train LogisticRegression and evaluate on both train and test.
    Class imbalance handled with class_weight='balanced' (no resampling).
    Returns a dict with train/test metrics plus ROC and confusion-matrix data.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    model = LogisticRegression(
        max_iter=1000,
        random_state=random_state,
        class_weight='balanced'   # handles ~14%/86% imbalance
    )
    model.fit(X_train, y_train)

    # Predictions on TRAIN and TEST
    y_pred_tr = model.predict(X_train)
    y_prob_tr = model.predict_proba(X_train)[:, 1]
    y_pred    = model.predict(X_test)
    y_prob    = model.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    return {
        'name'        : name,
        'train_size'  : len(X_train),
        'test_size'   : len(X_test),
        'train_acc'   : accuracy_score(y_train, y_pred_tr),
        'train_f1'    : f1_score(y_train, y_pred_tr, zero_division=0),
        'train_auc'   : roc_auc_score(y_train, y_prob_tr),
        'accuracy'    : accuracy_score(y_test, y_pred),
        'precision'   : precision_score(y_test, y_pred, zero_division=0),
        'recall'      : recall_score(y_test, y_pred, zero_division=0),
        'f1'          : f1_score(y_test, y_pred, zero_division=0),
        'roc_auc'     : roc_auc_score(y_test, y_prob),
        'fpr'         : fpr,
        'tpr'         : tpr,
        'y_test'      : y_test,
        'y_pred'      : y_pred,
        'model'       : model,
    }

print("✅ `train_evaluate` defined")
print("   · LogisticRegression with class_weight='balanced' (no resampling)")
print("   · Train AND test evaluated to measure the overfitting gap")

In [ ]:
# ── CELL 6: Run the three splits ──────────────────────────────────────────────
results = [
    train_evaluate(X_scaled, y, test_size=0.30, name="Split 70/30"),
    train_evaluate(X_scaled, y, test_size=0.60, name="Split 40/60"),
    train_evaluate(X_scaled, y, test_size=0.20, name="Split 80/20"),
]

print("=" * 78)
print(f"{'COMPARATIVE METRICS — BINARY LOGISTIC REGRESSION':^78}")
print("=" * 78)
header = (f"{'Split':<12} {'NTrain':>7} {'NTest':>7} "
         f"{'TrAcc':>7} {'TeAcc':>7} {'TrF1':>7} {'TeF1':>7} "
         f"{'TrAUC':>7} {'TeAUC':>7}")
print(header)
print("-" * 78)
for r in results:
    print(
        f"{r['name']:<12} {r['train_size']:>7,} {r['test_size']:>7,}"
        f" {r['train_acc']:>7.4f} {r['accuracy']:>7.4f}"
        f" {r['train_f1']:>7.4f} {r['f1']:>7.4f}"
        f" {r['train_auc']:>7.4f} {r['roc_auc']:>7.4f}"
    )
print("=" * 78)
print("Tr = Train · Te = Test")

## V. Visualizations

In [ ]:
# ── CELL 7: Comparative ROC curves ────────────────────────────────────────────
COLORS = ['#1e3c72', '#27ae60', '#e67e22']

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random classifier (AUC=0.50)')

for r, color in zip(results, COLORS):
    ax.plot(
        r['fpr'], r['tpr'], color=color, lw=2.2,
        label=f"{r['name']} — AUC = {r['roc_auc']:.4f}"
    )

ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('True Positive Rate (TPR)', fontsize=12)
ax.set_title('ROC Curves — Binary Logistic Regression\n'
             'Predicting High Quality Wine (quality >= 7)', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()
print("✅ ROC curves generated")

In [ ]:
# ── CELL 8: Confusion Matrices (3 splits) ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices — Binary Logistic Regression — Wine Quality',
             fontsize=13, fontweight='bold')

for ax, r, color in zip(axes, results, COLORS):
    cm   = confusion_matrix(r['y_test'], r['y_pred'])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Standard (0)', 'High quality (1)']
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{r['name']}\nAccuracy={r['accuracy']:.4f}", fontsize=11)

plt.tight_layout()
plt.show()
print("✅ Confusion matrices generated")

In [ ]:
# ── CELL 9: Test metric comparison ────────────────────────────────────────────
metric_names = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
labels       = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

x     = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
for i, (r, color) in enumerate(zip(results, COLORS)):
    vals = [r[m] for m in metric_names]
    bars = ax.bar(x + i * width, vals, width, label=r['name'],
                  color=color, alpha=0.88)
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim([0, 1.12])
ax.set_ylabel('Metric value (TEST)', fontsize=11)
ax.set_title('Test Metric Comparison — Three Splits\n'
             'Binary Logistic Regression — Wine Quality',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(y=1.0, color='#ccc', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()
print("✅ Comparative metrics chart generated")

In [ ]:
# ── CELL 10: Coefficient importance (Split 80/20) ─────────────────────────────
ref_model = results[2]['model']   # 80/20 split
coefs     = pd.Series(ref_model.coef_[0], index=X_scaled.columns)
coefs_ord = coefs.abs().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Coefficients — Logistic Regression (Split 80/20) — Wine Quality',
             fontsize=13, fontweight='bold')

coefs_ord.plot(kind='barh', ax=axes[0], color='#2a5298', alpha=0.85)
axes[0].set_xlabel('|Coefficient| — relative importance', fontsize=11)
axes[0].set_title('Importance (absolute value)', fontweight='bold')
axes[0].axvline(x=0, color='black', linewidth=0.8)
for patch in axes[0].patches:
    w = patch.get_width()
    if w > 0.01:
        axes[0].text(w + 0.01, patch.get_y() + patch.get_height() / 2,
                     f'{w:.3f}', va='center', fontsize=9)

coefs_signed = coefs.sort_values()
colors_sign  = ['#e74c3c' if v > 0 else '#3498db' for v in coefs_signed]
coefs_signed.plot(kind='barh', ax=axes[1], color=colors_sign, alpha=0.85)
axes[1].set_xlabel('Coefficient (signed)', fontsize=11)
axes[1].set_title('Effect direction\n🔴 positive -> increases P(high quality)\n🔵 negative -> decreases P(high quality)',
                  fontweight='bold', fontsize=10)
axes[1].axvline(x=0, color='black', linewidth=1.5)

plt.tight_layout()
plt.show()

print("✅ Coefficients generated")
print(f"\n📊 Top 5 variables with strongest positive influence (increase P(high quality)):")
print(coefs.sort_values(ascending=False).head(5).to_string())
print(f"\n📊 Top 5 variables with strongest negative influence (decrease P(high quality)):")
print(coefs.sort_values(ascending=True).head(5).to_string())

## VI. Final Metrics Table and Conclusions

In [ ]:
# ── CELL 11: Summary DataFrame of metrics (train + test) ──────────────────────
summary = pd.DataFrame([{
    'Split'      : r['name'],
    'N Train'    : r['train_size'],
    'N Test'     : r['test_size'],
    'Train Acc'  : round(r['train_acc'], 4),
    'Test Acc'   : round(r['accuracy'],  4),
    'Train F1'   : round(r['train_f1'],  4),
    'Test F1'    : round(r['f1'],        4),
    'Train AUC'  : round(r['train_auc'], 4),
    'Test AUC'   : round(r['roc_auc'],   4),
    'Precision'  : round(r['precision'], 4),
    'Recall'     : round(r['recall'],    4),
} for r in results])

summary.set_index('Split', inplace=True)

print("=" * 70)
print(f"{'FINAL METRICS TABLE (train + test)':^70}")
print("=" * 70)
try:
    display(summary.style
        .background_gradient(cmap='Greens',
                             subset=['Test Acc', 'Test F1', 'Test AUC', 'Precision', 'Recall'])
        .format('{:.4f}',
                subset=['Train Acc','Test Acc','Train F1','Test F1','Train AUC','Test AUC','Precision','Recall'])
        .set_caption("Binary Logistic Regression — Predicting High Quality Wine (quality >= 7)")
    )
except:
    print(summary.to_string())

# Overfitting gap check — expected to be small for Logistic Regression
print("\n📌 Overfitting gap (Train F1 - Test F1):")
for r in results:
    gap = r['train_f1'] - r['f1']
    flag = "✅ healthy" if gap < 0.10 else ("⚠️ moderate" if gap < 0.20 else "❌ high")
    print(f"   {r['name']}: {gap:+.4f}  {flag}")
print("\n   If all gaps are small (✅), Logistic Regression does NOT overfit here,")
print("   which is why oversampling/resample is not needed in this notebook.")

## VII. Conclusions

### What did we learn?

| Aspect | Observation |
|---------|------------|
| **Model stability** | If the three splits yield similar metrics, the model generalizes well and does not depend on the luck of a specific partition. |
| **Overfitting** | The train–test gap is reported per split. For Logistic Regression (implicit L2 regularization, linear boundary, 11 coefficients) the gap is small, so **no resampling is needed here**. |
| **Accuracy vs AUC** | Accuracy can be misleading given the ~14%/86% imbalance. ROC-AUC is the more robust metric here. |
| **Precision vs Recall** | There is a trade-off: raising the decision threshold increases Precision but lowers Recall. |
| **Coefficients** | `alcohol` (positive) and `volatile acidity` (negative) dominate the prediction, consistent with the EDA and Spearman correlation. |
| **Class imbalance** | Handled with `class_weight='balanced'` — a separate concern from overfitting. |
| **Model limitation** | Logistic Regression assumes a linear relationship between variables and log-odds. Non-linear interactions require more complex models. |

### Comparison with the CART Decision Tree

| | Logistic Regression | CART Tree |
|---|---|---|
| **Scaling** | Required (StandardScaler) | Not required |
| **Decision boundary** | Linear | Non-linear (piecewise) |
| **Variable importance** | Coefficients (log-odds) | Gini impurity reduction |
| **Overfitting risk** | Low (implicit regularization) → no resampling | High if max_depth not controlled → resample applied |
| **Interpretability** | Logistic equation | If-else rule tree |

### Suggested next steps
- Tune the classification threshold (default 0.5) according to priority (Recall vs Precision).
- Evaluate whether changing the binarization threshold (quality >= 7 → quality >= 6) improves class balance.
- Apply cross-validation (k-fold) for a more robust metric estimate.